# 06 — Carga das tabelas no banco PostgreSQL (Supabase)

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Destino:** PostgreSQL gerenciado no Supabase (região São Paulo), via conexão *Session pooler* (compatível com IPv4) e SSL obrigatório.

**Credenciais:** lidas de `.streamlit/secrets.toml`, arquivo ignorado pelo Git — nenhuma senha aparece neste notebook.

**Carga:** as 9 tabelas de `data/processed/`, com conferência de contagem de linhas entre CSV e banco.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"
SECRETS = ROOT / ".streamlit" / "secrets.toml"

# Lê as credenciais do arquivo protegido (nunca escritas no código)
with open(SECRETS, "rb") as arquivo:
    cfg = tomllib.load(arquivo)["connections"]["postgresql"]

url = URL.create(
    drivername="postgresql+psycopg2",
    username=cfg["username"],
    password=cfg["password"],
    host=cfg["host"],
    port=int(cfg["port"]),
    database=cfg["database"],
    query=cfg.get("query", {}),
)
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conexao:
    versao = conexao.execute(text("SELECT version()")).scalar()
print("Conexão estabelecida:", str(versao).split(",")[0])

Conexão estabelecida: PostgreSQL 17.11 on aarch64-unknown-linux-gnu


In [2]:
TABELAS = [
    "dim_empresa", "fato_macro", "fato_macro_mensal", "fato_macro_anual",
    "fato_cotacoes", "fato_mercado_anual", "fato_contabil",
    "fato_indicadores_contabeis", "fato_anual_integrado",
]
COLUNAS_DATA = {"fato_macro": ["data"], "fato_macro_mensal": ["mes"], "fato_cotacoes": ["data"]}
COLUNAS_TEXTO = {"cnpj": "str", "cd_conta": "str"}

resumo = []
for nome in TABELAS:
    df = pd.read_csv(PROC_DIR / f"{nome}.csv",
                     parse_dates=COLUNAS_DATA.get(nome, []), dtype=COLUNAS_TEXTO)
    df.to_sql(nome, engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    resumo.append({"tabela": nome, "linhas_csv": len(df)})
    print(f"{nome}: {len(df)} linhas carregadas")

dim_empresa: 9 linhas carregadas
fato_macro: 3201 linhas carregadas
fato_macro_mensal: 60 linhas carregadas
fato_macro_anual: 5 linhas carregadas
fato_cotacoes: 11169 linhas carregadas
fato_mercado_anual: 45 linhas carregadas
fato_contabil: 9894 linhas carregadas
fato_indicadores_contabeis: 45 linhas carregadas
fato_anual_integrado: 45 linhas carregadas


In [3]:
with engine.connect() as conexao:
    for item in resumo:
        consulta = text(f'SELECT COUNT(*) FROM "{item["tabela"]}"')
        item["linhas_banco"] = conexao.execute(consulta).scalar()

conferencia = pd.DataFrame(resumo)
conferencia["confere"] = conferencia["linhas_csv"] == conferencia["linhas_banco"]
conferencia

,tabela,linhas_csv,linhas_banco,confere
0,dim_empresa,9,9,True
1,fato_macro,3201,3201,True
2,fato_macro_mensal,60,60,True
3,fato_macro_anual,5,5,True
4,fato_cotacoes,11169,11169,True
5,fato_mercado_anual,45,45,True
6,fato_contabil,9894,9894,True
7,fato_indicadores_contabeis,45,45,True
8,fato_anual_integrado,45,45,True


In [4]:
consulta = text("""
    SELECT ticker, setor, ROUND((roe * 100)::numeric, 1) AS roe_pct
    FROM fato_anual_integrado
    WHERE ano = 2023 AND roe IS NOT NULL
    ORDER BY roe DESC
""")
pd.read_sql(consulta, engine)

,ticker,setor,roe_pct
0,EGIE3,Energia,34.9
1,WEGE3,Exportadoras Industriais,32.9
2,PETR4,Energia,32.7
3,SUZB3,Commodities Exportadoras,31.5
4,VALE3,Commodities Exportadoras,20.4
5,ABEV3,Consumo/Varejo,18.7
6,ITUB4,Bancos,17.0
7,LREN3,Consumo/Varejo,9.7
8,EMBR3,Exportadoras Industriais,5.3


## Conclusão da carga

| Verificação | Resultado |
|---|---|
| Conexão | PostgreSQL 17 (Supabase, São Paulo) via Session pooler, SSL obrigatório |
| Credenciais | Lidas de `.streamlit/secrets.toml` (ignorado pelo Git); nenhuma senha no código ou nas saídas |
| Tabelas carregadas | 9 de 9 |
| Integridade | Contagem de linhas idêntica entre CSV e banco em todas as tabelas |
| Teste de consulta | Ranking de ROE 2023 executado diretamente no PostgreSQL |